# H&M NGCF · seed49 · M1/M2/M3/M4/M5_A/B/C/D

기존 H&M 개입 조건을 유지하고 NGCF 백본에서 8모형을 각각 새로 학습합니다. LightGCN 참조는 입력 검증에만 쓰며 NGCF 성능으로 재사용하지 않습니다.

- train ~2020-09-15 / test 2020-09-16~22, 신규상품 추천, MIN_ITEM_INTER=1.
- seed49, 모형당300epoch 마지막점만 평가. validation·holdout·조기종료·중간 test·test 기반 재튜닝 없음.
- ID64, 2층64, 층 연결 출력192, message dropout0.1, node dropout0. batch131072, Adam lr0.0005, ID L2 0.001, M2/백본 L2 각각1e-6.
- 기존 프로젝트 NGCF: `h_next = dropout(LeakyReLU(W1(h+A h)+W2(h*(A h))),0.1)`. 층별 정규화한 표현과 초기 ID를 연결합니다. 다음 층 입력은 정규화 전 h_next입니다.
- M3는 기존 중심화 N/V 가중치의 **대칭 차수 정규화**를 사용합니다. GraphSAGE 행별 평균을 사용하지 않습니다.
- M2의 동일 입력·양성상품 구매자 제외·적응형 직교 결합을 유지하며 encoder 출력만 NGCF 표현에 맞춰192입니다. 단일 optimizer 공동학습, 사전학습·동결 없음.
- M4 및 조합은 기존 H&M 정의 그대로입니다. CLV는 historical CLV proxy입니다.

## 저장공간
**GraphSAGE와 병행하면 여유 있는 개인 Drive에 이 실행을 저장하는 구성을 권장합니다.** GPU/Drive 로그인 계정을 확인하세요. 학교 Drive 10GB에서는 두 실험의 체크포인트 교체가 겹치면 부족할 수 있습니다.

입력은 검증 후 런타임 디스크로 복사합니다. 진행 중 모형은 매epoch 전체 모델·Adam·난수를 Drive에 저장합니다. 완료 모형은 최종 가중치·이력·난수를 보존하고 Adam만 제외합니다. NGCF 체크포인트는 완료 약3GB, 교체 중 최고 약5GB 예상이며 실제 크기는 기록합니다. 결과 ZIP에는 가중치를 중복 포함하지 않습니다.

아래 셀은 Drive API 잔여 용량을 확인하며 기본적으로 NGCF용6GB와 다른 실험용5GB 여유를 요구합니다. 부족하면 학습 전에 중단하며 기존 파일을 삭제하지 않습니다. 매epoch 저장 전에도 검사합니다. 다른 앱의 동시 쓰기와 서버 반영 지연까지 절대 보장할 수는 없으므로 충분한 여유 공간을 확보하세요. **같은 Drive의 GraphSAGE가 계속 학습 중이면 여유분 설정을 줄이지 마세요.**

## 실행·재개
L4 GPU와 고용량 RAM에서 위에서부터 실행합니다. 원본 거래·상품과 참조 ZIP 4개는 기존 `MyDrive/논문/data` 경로를 사용합니다. 입력이 없는 계정으로 바꿨다면 동일 원본 4개를 해당 경로에 준비해야 합니다. 잘못 변환된 기존 5조각 ZIP을 사용하지 마세요.

끊기면 같은 링크·출력 경로로 재실행합니다. 완료 모형은 건너뛰고 미완료 모형은 마지막 저장 epoch 다음부터 이어갑니다. 같은 출력 폴더를 두 런타임에서 동시에 실행하지 마세요. 최종 평가 후 결과 파일 기록 전에 끊기면 최종 평가를 다시 계산할 수 있습니다.

CPU 합성자료에서8모형 최적화 전후 학습·가중치·Adam·난수의 정확 일치와 재개를 확인했습니다. **실제 H&M NGCF 전체 GPU 학습·메모리·소요시간은 아직 미검증**입니다. 첫 배치와 epoch 계측을 확인하세요. 1시드이므로 표준편차·신뢰구간·유의성을 주장하지 않으며 약10시드 최종평균을 대체하지 않습니다.

모형 정의 참고: [NGCF 원논문](https://arxiv.org/abs/1905.08108). 프로젝트의 ID/BPR/정규화 설정을 유지한 구현이며 원본 TensorFlow 벤치마크 전체 재현은 아닙니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '836c0c71c6aab3a952f8f948f9920b32c7b109c8'
REPO = Path('/content/clv-hm-ngcf-v1-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 저장공간 사전 확인
Drive 연결과 동일한 계정으로 인증하세요. 용량 부족 시 입력 준비·학습 전에 중단합니다.

In [ ]:
# Authenticate with the SAME account selected for drive.mount above.
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
OTHER_RUN_RESERVE_GB = 5.0  # Keep while GraphSAGE shares this Drive.

def check_space(required_bytes):
    info = drive_api.about().get(fields='user(emailAddress),storageQuota').execute()
    quota = info.get('storageQuota', {})
    if 'limit' not in quota or 'usage' not in quota:
        raise RuntimeError('계정 잔여 용량을 확인할 수 없습니다. 학습을 시작하지 않습니다.')
    available = int(quota['limit']) - int(quota['usage'])
    needed = required_bytes + int(OTHER_RUN_RESERVE_GB * 10**9)
    if available < needed:
        raise RuntimeError(f"Drive 공간 부족 예방: {info['user']['emailAddress']} / "
                           f"남음 {available/1e9:.2f}GB, 필요한 여유 {needed/1e9:.2f}GB. "
                           '여유 있는 개인 Drive로 실행하거나 다른 실험 결과를 안전하게 보관한 뒤 재개하세요.')
    return info['user']['emailAddress'], available

account, available = check_space(6 * 10**9)
print('용량 확인 계정:', account, '| 잔여 GB:', round(available / 1e9, 2))
# Verify that the mounted MyDrive and quota API belong to the same account.
import uuid, time
probe = Path('/content/drive/MyDrive') / ('.clv_ngcf_account_' + uuid.uuid4().hex)
probe.write_text('NGCF account verification')
try:
    found = []
    for attempt in range(10):
        found = drive_api.files().list(q=f"name = '{probe.name}' and trashed = false and 'me' in owners",
                                      fields='files(id)', pageSize=10).execute().get('files', [])
        if found: break
        time.sleep(2)
    assert len(found) == 1, 'Drive 마운트와 용량 확인 계정이 다르거나 동기화가 지연되었습니다. 같은 계정으로 연결 후 재시도하세요.'
finally:
    probe.unlink(missing_ok=True)  # Only this newly created verification marker.


## 1. 복구된 연결된 Drive 입력 검증 — 학습 없음
원본 거래·상품과 M2/M4 참조 ZIP의 SHA256을 검사합니다. 기존 5조각 ZIP을 재설치하지 않습니다. 전체 전처리 후 M2 특징·M3 그래프·M4 가중치도 기존 실험과 대조합니다.


In [ ]:
import hashlib
DATA = Path('/content/drive/MyDrive/논문/data')
EXPECTED_FILES = {
    'raw/hm/transactions_train.parquet': '0d90884fc1bc400161e6aeba238cba94183998bedfeb4ebbf4962e17d0c8489b',
    'raw/hm/articles.csv': '25f21882053517f7684ea43f524225a9481a7bb9aab9cf0abfc7fe8c4576a72b',
    'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip': '6d5159a888319b8a443a6907922a6e36fe9954c6b717dc2b1a6ff547c1d52bed',
    'references/clv_hm_m4c_lastweek_seed49_results.zip': '11fbfbddd21f966357cc872d43891bd078741b83037e88ee39a5558498ce40fe',
}
for name, expected in EXPECTED_FILES.items():
    path = DATA / name
    assert path.is_file(), f'연결된 Drive 입력 파일 없음: {path}'
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b''):
            digest.update(block)
    assert digest.hexdigest() == expected, f'원본과 다른 입력: {name}. 이전 ZIP을 재설치하지 마세요.'
print('연결된 Drive 원본 입력·참조 4개 SHA256 일치')

import clv_hm_ngcf_final8_seed49 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이전 모듈이 메모리에 남아 있습니다. 이 런타임만 다시 시작한 후 위에서부터 실행하세요.'
# Fixed raw bytes are read from the runtime disk, not repeatedly from Drive.
import shutil
LOCAL_INPUT = Path('/content/hm_ngcf_inputs')
LOCAL_INPUT.mkdir(exist_ok=True)
for key, name in [('tx_path', 'transactions_train.parquet'), ('item_meta_path', 'articles.csv')]:
    source = DATA / 'raw/hm' / name
    target = LOCAL_INPUT / name
    expected = EXPECTED_FILES['raw/hm/' + name]
    def matches(path):
        if not path.is_file():
            return False
        h = hashlib.sha256()
        with path.open('rb') as f:
            for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
                h.update(block)
        return h.hexdigest() == expected
    if not matches(target):
        temporary = target.with_suffix(target.suffix + '.partial')
        shutil.copyfile(source, temporary)
        assert matches(temporary), f'로컬 복사 해시 불일치: {name}'
        temporary.replace(target)
    screen.base.v3.SCHEMA['hm'][key] = str(target)
print('원본과 동일한 Colab 로컬 입력 준비 완료')
cfg = screen.configure(
    m2_reference=str(DATA / 'references/clv_m2_adaptive_hm_seed49_lastweek_results.zip'),
    m4_reference=str(DATA / 'references/clv_hm_m4c_lastweek_seed49_results.zip'),
    out_dir=str(DATA / 'results_v3_hm_ngcf_final8_seed49_v1'),
)
screen.STORAGE_GUARD = check_space
print('출력:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('검증 완료:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])


## 2. NGCF 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 NGCF M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('NGCF M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/hm_ngcf_final8_seed49_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))